# Bank Churn (Logistic Regression)

## Introduction

**Dataset**: https://www.kaggle.com/datasets/santoshd3/bank-customers

It is advantageous for banks to know what leads a client towards the decision to leave the company. Churn prevention allows companies to develop loyalty programs and retention campaigns to keep as many customers as possible.

This dataset contains some customers who are withdrawing their account from the bank. With this dataset, we try to analyse and predict which customers have a high chance of leaving the bank.

## Import Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Import Dataset

In [2]:
df = pd.read_csv("../data/Churn_Modelling.csv")

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


## Exploratory Data Analysis (EDA)

EDA is the first step in a data science or machine learning project. It involves exploring, visualizing, and summarizing your data to understand what is in the dataset, what problems or patterns are there and what steps are needed before modeling. Common Steps are:

- Understand the structure of the dataset
- Check for missing values and duplicate rows
- Visualize distributions
- Check for correlations
- Visualize relationships
- Outlier detection

## Checking for Duplicates and Missing Values

In [4]:
df.duplicated().sum()

np.int64(0)

Duplicate rows can skew the learning process. If the same data point appears multiple times, the model may treat it as more important than it really is. We would rather drop these.

Missing values are also a problem.

Most ML algorithms cannot handle undefined values i.e NaN. If there are any missing values in the dataset we typically handle them by:

- **Dropping rows/columns** (if the proportion of missing values is small or irrelevant)

- **Imputing values** using statistical techniques (mean, median, mode, etc.)

In [5]:
df.isna().any().any()

np.False_

## Dropping Unnecessary Columns

Some columns are unimportant from the get-go. Some may be are arbitrary identifiers (RowNumber, CustomerID) with no relationship to target. They may also be sensitive personal information (Surname) or, the feature is constant or near constant (for example, if 99.9% of the participants came from France then that feature would not be useful for training).

In more practical settings, you may consider whether a feature will be available when making real predictions.

In [6]:
df.drop(columns=["RowNumber", "CustomerId", "Surname"], inplace=True)

In [7]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Encoding Categorical Columns

There are two main types of categorical data:
1. **Nominal Categorical Data:** Categories with no inherent order. You cannot rank them meaningfully.

    Examples:
    - Color: Red, Blue, Green
    - Country: Nigeria, Ghana, Kenya
    - Gender: Male, Female


2. **Ordinal Categorical Data:** Categories with a clear, meaningful order.

    Examples:
    - Education level: High school < Bachelor < Master
    - Satisfaction rating: Poor < Average < Good < Excellent
    - Size: Small < Medium < Large

Most ML algorithms only work with numerical inputs. You cannot directly feed text labels like “Red” or “Master’s” into a model. It will not understand them. So, we convert (or *encode*) them into numbers using different strategies depending on the type of category.

The common encoding methods are *label encoding* and *one-hot encoding*.

**Label Encoding**

Here, each category is assigned a unique integer that represents the categories (0, 1, 2,...). We use it for *ordinal* data where we want to preserve the order/rank between values. You can also use it for binary categories (e.g. Male/Female) because with only two values, any implied ordering is irrelevant.

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Gender"] = le.fit_transform(df["Gender"])

In [9]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


**One-Hot Encoding**

This creates a new binary (0 or 1) column for each category. For example, if there are 3 categories (France, Germany and Spain), you get 3 new columns. It is used for *nominal* data (no order) where you want to avoid introducing false relationships.

In [10]:
df = pd.get_dummies(df, columns=["Geography"])

In [11]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,True,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,0,False,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,1,True,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,0,True,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,0,False,False,True


## Getting our Features and Targets

Our target (dependent variable) is the "Exited" column. So everything else will be our features (independent variables).

In [12]:
X = df.drop(columns=["Exited"])
y = df["Exited"]

## Splitting into Train and Test Sets

When building a machine learning model, we want to know how well it will perform on new, unseen data, not just the data we used to train it. To do that, we split the dataset into two parts:
1. **Training Set**
    - This is the data the model learns from.

2. **Test Set**
    - This is held back from the model during training.
    - It’s used to evaluate the model’s performance on unseen data.
    - Helps check for *overfitting*; when a model does well on training data but fails to generalize.

Typically, we use a split of **80-20**. 80% of the data for the training set and the remaining 20% for the test set.

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

## Scaling Numerical Features

Logistic regression uses gradient descent, which is sensitive to feature scales so, features with larger ranges can dominate the optimization process. We have many kinds of scaling methods:

**StandardScaler (Z-score normalization):** Best when features are normally distributed. The data is scaled in such a way that the Mean = 0 and Standard Deviation = 1.
$$
\frac{x - \text{mean}}{\text{std}}
$$

**MinMaxScaler:** Scales to range [0,1]. It is good when you know the approximate upper/lower bounds.
$$
\frac{x - \text{min}}{\text{max} - \text{min}}
$$

**RobustScaler:** Uses median and interquartile range. It works best when the data has outliers as it is less sensitive to extreme values.
$$
\frac{x - \text{median}}{\text{IQR}}
$$

**Important:** Fit scaler on training data only, then transform both train and test sets. You should never let your test data "leak" into the training process.

In [14]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]

scaler = StandardScaler()

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

## Training the Model

Here, we simply initialize our model and feed it our training set.

In [15]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression()
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

## Evaluating the Model

Once you have trained a classification model, you need to measure how well it performs on unseen data (test set).

You do this using evaluation metrics that tell you how accurate or useful your predictions are. Especially for real-world decisions like spam detection, fraud detection, disease diagnosis, etc.

In [16]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score


y_pred = model.predict(X_test)

**NOTES**

|                         | Meaning                                  | 
| ----------------------- | ---------------------------------------- |
| **TP** (True Positive)  | Model correctly predicted **Positive**   |
| **TN** (True Negative)  | Model correctly predicted **Negative**   |
| **FP** (False Positive) | Model incorrectly predicted **Positive** |
| **FN** (False Negative) | Model incorrectly predicted **Negative** |

### Confusion Matrix

Say we test some people for the presence of a disease. Some of these people have the disease, and our test correctly says they are positive. They are called true positives (TP). Some have the disease, but the test incorrectly claims they don't. They are called false negatives (FN). Some don't have the disease, and the test says they don't – true negatives (TN). Finally, there might be healthy people who have a positive test result – false positives (FP). These can be arranged into a 2×2 **confusion matrix**, conventionally with the test result on the vertical axis and the actual condition on the horizontal axis.

|                      | **Predicted Positive** | **Predicted Negative** |
|----------------------|------------------------|------------------------|
| **Actual Positive**  | True Positive (TP)     | False Negative (FN)    |
| **Actual Negative**  | False Positive (FP)    | True Negative (TN)     |


In [17]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1544   63]
 [ 314   79]]


### Accuracy Score

The **accuracy score** is a metric used to evaluate the performance of a model by measuring the proportion of correct predictions made out of the total predictions.

$$
\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Number of Predictions}}
$$
or

$$
\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}
$$

In [18]:
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.8115

### Precision Score

The **precision score** is a metric used in machine learning to measure the proportion of true positive predictions among all positive predictions made by a model. In other words, how many predicted positives were actually correct.

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}
$$

It is important when false positives are costly e.g., flagging someone as a fraudster when they are not.

In [19]:
precision = precision_score(y_test, y_pred)

precision

0.5563380281690141

### Recall Score

The **recall score** is a measure of how many actual positives did the model catch.

$$
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}
$$
It is important when false negatives are costly e.g., missing a cancer diagnosis.

In [20]:
recall = recall_score(y_test, y_pred)

recall

0.2010178117048346

### F1 Score

The **F1 score** is a performance metric used in machine learning that represents the harmonic mean of precision and recall. It is for when you care about both precision and recall.

$$
\text{F1 Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

In [21]:
f1 = f1_score(y_test, y_pred)

f1

0.2953271028037383

## Extra: Overfitting and Underfitting

When training a machine learning model, our goal is not just to make accurate predictions on the training data, but also to ensure that it performs well on unseen data. This is called **generalization**. Machine Learning models are "good" if:
- It learns patterns effectively from the training data.
- It generalizes well to new, unseen data.

Two common problems that prevent generalization are:
1. Overfitting
2. Underfitting

### Overfitting

**Overfitting** happens when a model learns too much from the training data, including details that do not matter (like noise or outliers). As a result, the model works great on training data but fails when tested on new data. 

Overfitting models are like students who memorize answers instead of understanding the topic. They do well in practice tests (training) but struggle in real exams (testing).

### Underfitting

**Underfitting** happens when a model is too simple to capture what is going on in the data. In this case, the model doesn’t work well on either the training or testing data.

Underfitting models are like students who don’t study enough. They do not do well in practice tests or real exams.

In [22]:
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print(f"Training Accuracy: {train_acc}")
print(f"Test Accuracy: {test_acc}")

Training Accuracy: 0.811
Test Accuracy: 0.8115


From the similar training and test accuracy scores, we can infer that the model is likely did not overfit nor underfit. 

## Assignment

**Dataset:** https://www.kaggle.com/datasets/yasserh/titanic-dataset

The dataset contains the information about the passengers who were abroad the Titanic during its sinking in 1912. Using this, create a predictive model to find out whether a passenger survived or not.